# CLP Transformer Training Pipeline

This notebook details the training process for the Transformer model. It covers loading the preprocessed trajectory datasets, defining the model architecture, configuring the training hyperparameters (including loss and evaluation metrics), and saving the finalized weights.

## 1. Dataset Loading

Loads the preprocessed trajectory data categorized into difficulty levels (easy, medium, and hard). These levels are defined by how the baseline VCS heuristic ranked the optimal action (e.g., Top-1 for easy, Top-2 to Top-8 for medium), allowing the model to learn to re-rank candidate blocks effectively when the heuristic falls short.

In [1]:
from data.preprocessing import load_dataset

dataset_easy = load_dataset("M30_1-1.data")
dataset_medium = load_dataset("M30_2-8.data")
dataset_hard = load_dataset("M30_9-64.data")

Dataset M30_1-1.data cargado con 10000 muestras.
Dataset M30_2-8.data cargado con 10000 muestras.
Dataset M30_9-64.data cargado con 10000 muestras.


## 2. Model Initialization

Defines the core hyperparameters for the Transformer architecture and initializes the model. The network is configured to process the static block features, the candidate actions, and the relative spatial coordinates of the already placed blocks.

In [ ]:
from models.clp_transformer import CLPTransformer

# Model hyperparameters
block_dim = 5
action_dim = 2
placed_dim = 6
d_model = 32
nhead = 4
num_layers = 2
ff_dim_multiplier = 2
dropout = 0.1

# Initialize model
model = CLPTransformer(
    block_dim=block_dim,
    action_dim=action_dim,
    placed_dim=placed_dim,
    d_model=d_model,
    nhead=nhead,
    num_layers=num_layers,
    ff_dim_multiplier=ff_dim_multiplier,
    dropout=dropout
)

## 3. Model Training

Configures and executes the training loop. This setup defines the learning rate scheduler (`LRConfig`), early stopping (`patience`), loss function (`CrossEntropyLoss`), and evaluation metrics (Accuracy@1, Accuracy@K, and Mean Reciprocal Rank). While the framework supports progressive curriculum learning phases via the `epochs` array, the current configuration (`[0, 0, 10]`) skips directly to the final phase. The model is trained immediately on a combined mixture of all three datasets, utilizing the `dataset_weights` parameter to explicitly weight the loss function according to the difficulty level of each sample.

In [ ]:
from training.training import train, CrossEntropyLoss, Accuracy, MeanReciprocalRank, LRConfig

# Configurable hyperparameters
epochs = [0, 0, 10]
datasets = [dataset_easy, dataset_medium, dataset_hard]
train_size = 800
test_size = 200
dataset_weights = [0.25, 0.5, 0.25]
batch_size = 64
lr_config = LRConfig(start=1e-4, factor=0.5, patience=10, min=1e-6)
weight_decay = 1e-5
patience = 20
seed = 42

# Loss function and metrics
loss_function = CrossEntropyLoss()
metrics = [Accuracy(), Accuracy(k=8), MeanReciprocalRank()]

# LR scheduling for each training phase
lr_configs = [LRConfig(start=1e-4), LRConfig(start=1e-4), lr_config]

model = train(
    model, epochs, datasets, train_size, test_size, 
    batch_size, lr_configs, weight_decay, loss_function, 
    dataset_weights, patience, metrics, seed
)

ℹ️ Usando dispositivo: cpu

ℹ️ Iniciando fase: 3/3

Epoch 1/10
    Global - Wgt Train Loss: 3.7061 | Wgt Val Loss: 3.7478
    [Train] M30_1-1.data - Loss: 3.4459
    [Train] M30_2-8.data - Loss: 3.6515
    [Train] M30_9-64.data - Loss: 4.0753
    [Val] M30_1-1.data - Loss: 3.5811 | Accuracy: 20.00% | Top-8 Accuracy: 51.00% | MRR: 0.311 | 
    [Val] M30_2-8.data - Loss: 3.6699 | Accuracy: 7.50% | Top-8 Accuracy: 48.50% | MRR: 0.231 | 
    [Val] M30_9-64.data - Loss: 4.0703 | Accuracy: 4.00% | Top-8 Accuracy: 30.00% | MRR: 0.139 | 
    [Avg] Accuracy: 10.50% | Top-8 Accuracy: 43.17% | MRR: 0.227 | 

Epoch 2/10
    Global - Wgt Train Loss: 3.6844 | Wgt Val Loss: 3.7269
    [Train] M30_1-1.data - Loss: 3.4187
    [Train] M30_2-8.data - Loss: 3.6248
    [Train] M30_9-64.data - Loss: 4.0691
    [Val] M30_1-1.data - Loss: 3.5526 | Accuracy: 21.00% | Top-8 Accuracy: 53.50% | MRR: 0.322 | 
    [Val] M30_2-8.data - Loss: 3.6481 | Accuracy: 7.50% | Top-8 Accuracy: 49.50% | MRR: 0.232 | 
    [Val]

## 4. Model Checkpointing

Saves the trained model weights and configuration to the disk, making it available for subsequent evaluation and inference in the validation pipeline.

In [6]:
from training.training import save_model

save_model(model, "example_model")

✅ Modelo guardado en /home/oscar/Escritorio/CLP-Framework/models/example_model.pth
